# BEE_HERo — Data is Training-Ready

This notebook executes the full **image "feature engineering"** layer and proves the dataset is 100% ready for a training loop:

1. Build the **class index** from `_pipeline/splits/split_assignments.csv` (label per image, contiguous `0..nc-1`).
2. Build the **augmentation + normalization** pipeline (Albumentations if installed, else an identical torchvision `v2` chain).
3. Build `Dataset` + `DataLoader`s tuned for **6 GB VRAM**.
4. **MixUp/CutMix** + soft-target loss.
5. Verify a real batch, visualize augmented images, run one **ResNet-18** train step, and (optionally) a short smoke training loop.

All heavy logic lives in `bee_hero_dataset.py`; this notebook just drives and visualizes it.

In [ ]:
import os, sys, time

# locate the project folder (where bee_hero_dataset.py lives)
ROOT = os.getcwd()
if not os.path.exists(os.path.join(ROOT, 'bee_hero_dataset.py')):
    ROOT = r'C:\Users\narim\Desktop\BEE_HERo'
sys.path.insert(0, ROOT)
print('project:', ROOT)

import torch
from bee_hero_dataset import (build_dataloaders, build_transforms, build_index,
                              mixup_cutmix, SoftTargetCrossEntropy, _denormalize)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1–3. Class index + transforms + DataLoaders
On GPU bump `batch_size` to 48–64 with AMP. `num_workers` is auto-set from CPU cores.

In [ ]:
train_dl, val_dl, test_dl, class_to_idx = build_dataloaders(
    root=ROOT, img_size=224, batch_size=32)
NC = len(class_to_idx)
print(f'num_classes = {NC}')
print(f'train batches = {len(train_dl)} | val = {len(val_dl)} | test = {len(test_dl)}')
_, backend = build_transforms(train=True)
print('augmentation backend:', backend)

## 4. Inspect & visualize one augmented training batch
Each call yields different crops/flips/colour/erasing — that is the on-the-fly "feature engineering".

In [ ]:
import matplotlib.pyplot as plt
import torchvision.utils as vutils

t0 = time.time()
xb, yb = next(iter(train_dl))
print(f'batch x={tuple(xb.shape)} dtype={xb.dtype}  y={tuple(yb.shape)} '
      f'in [{int(yb.min())},{int(yb.max())}]  ({time.time()-t0:.1f}s)')

grid = vutils.make_grid(_denormalize(xb[:16]), nrow=4)
plt.figure(figsize=(8, 8)); plt.axis('off')
plt.title('Augmented training batch (denormalized)')
plt.imshow(grid.permute(1, 2, 0).numpy()); plt.show()

## 5. MixUp / CutMix + soft-target loss

In [ ]:
xm, ym = mixup_cutmix(xb.clone(), yb, NC)
print('mixed x:', tuple(xm.shape), '| soft target:', tuple(ym.shape),
      '| row-sum ~', round(float(ym.sum(1).mean()), 3))

grid = vutils.make_grid(_denormalize(xm[:16]), nrow=4)
plt.figure(figsize=(8, 8)); plt.axis('off')
plt.title('After MixUp/CutMix'); plt.imshow(grid.permute(1, 2, 0).numpy()); plt.show()

## 6. One real training step (proves end-to-end readiness)

In [ ]:
import torchvision
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = torchvision.models.resnet18(weights=None, num_classes=NC).to(device)
criterion = SoftTargetCrossEntropy()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

model.train()
x, y = xm.to(device), ym.to(device)
out = model(x)
loss = criterion(out, y)
loss.backward(); optimizer.step(); optimizer.zero_grad()
print(f'logits={tuple(out.shape)}  loss={loss.item():.3f}  '
      f'(random-init reference = ln({NC}) = {torch.log(torch.tensor(float(NC))):.2f})')

## 7. (Optional) short smoke training loop
A handful of steps to confirm the loop runs and loss moves. On CPU keep it tiny; on GPU this becomes your real loop. Set `N_STEPS` higher to actually train.

In [ ]:
N_STEPS = 10  # raise this (or loop over epochs) for real training
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

model.train()
it = iter(train_dl)
for step in range(N_STEPS):
    try:
        xb, yb = next(it)
    except StopIteration:
        it = iter(train_dl); xb, yb = next(it)
    xb, ym = mixup_cutmix(xb.to(device), yb.to(device), NC)
    optimizer.zero_grad()
    with torch.autocast(device_type=device, enabled=use_amp):
        loss = criterion(model(xb), ym)
    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
    print(f'step {step+1:02d}/{N_STEPS}  loss={loss.item():.3f}')
print('smoke loop OK — plug in epochs, a scheduler, and validation to train for real.')

## ✅ Summary
- `train` / `val` / `test` DataLoaders yield normalized `224×224` tensors with valid `0..nc-1` labels.
- Augmentation (crop/flip/affine/colour/erasing) + ImageNet normalization run on-the-fly.
- MixUp/CutMix + soft-target CE wired in.
- A ResNet-18 train step runs end-to-end.

**The data is 100% ready for training.** Swap ResNet-18 for your backbone, add an epoch loop + LR scheduler + validation/accuracy, and train.